# Lab 1: Check your setup and meet the teacher and the student

In this lab you will:

1. Check that this machine can run the course labs.
2. Run the small **student** model and measure how fast it is.
3. Run the large **teacher** model and measure it the same way.
4. Compare the two.

Nothing is trained in this lab. It takes about 5 to 10 minutes once the models are downloaded.
The first run downloads about 16 GB of model files.

Run the cells in order, from top to bottom.

## 1. Check the machine

The cell below prints what it found. If it says **Not ready**, fix each line it lists before going on.
Lesson 1.6 on the course site explains what each problem means.

In [ ]:
from kd_course.env import check_environment

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

## 2. The prompts

Both models answer the same three customer messages. They come from the kind of task this course
keeps returning to: sorting banking support requests.

In [ ]:
PROMPTS = [
    "A customer writes: 'My new card still has not arrived after two weeks.' "
    "In one short sentence, say what the customer needs.",
    "A customer writes: 'I was charged twice for the same purchase yesterday.' "
    "In one short sentence, say what the customer needs.",
    "A customer writes: 'What exchange rate do you use when I pay abroad?' "
    "In one short sentence, say what the customer needs.",
]
MAX_NEW_TOKENS = 48

## 3. Run the student

The student has about 0.5 billion weights.

In [ ]:
from kd_course.bench import benchmark_generation, generate_text
from kd_course.models import STUDENT_ID, load_causal_lm

student, student_tokenizer = load_causal_lm(STUDENT_ID)
student_answers = [generate_text(student, student_tokenizer, p, MAX_NEW_TOKENS) for p in PROMPTS]
student_result = benchmark_generation(student, student_tokenizer, PROMPTS, MAX_NEW_TOKENS)

for answer in student_answers:
    print("-", answer)
print(student_result)

## 4. Run the teacher

The teacher has about 7.6 billion weights and needs most of the GPU's memory, so the student is
removed from the GPU first.

In [ ]:
import gc

import torch

from kd_course.models import TEACHER_ID

del student
gc.collect()
torch.cuda.empty_cache()

teacher, teacher_tokenizer = load_causal_lm(TEACHER_ID)
teacher_answers = [generate_text(teacher, teacher_tokenizer, p, MAX_NEW_TOKENS) for p in PROMPTS]
teacher_result = benchmark_generation(teacher, teacher_tokenizer, PROMPTS, MAX_NEW_TOKENS)

for answer in teacher_answers:
    print("-", answer)
print(teacher_result)

## 5. Compare

In [ ]:
from kd_course.bench import format_comparison

print(format_comparison(teacher_result, student_result))

## 6. What did you see?

Double-click this cell and write your answers under each question.

**a. How many times smaller is the student, and how many times faster? Are those two numbers the same? Why might they differ?**

**b. Read the two sets of answers side by side. Where is the student's answer worse than the teacher's?**

**c. The student is faster and cheaper but less accurate. What would have to be true for it to be good enough for this task?**

Question c is what the rest of the course is about: closing that gap for one task, and measuring it.